# Documentrelaties & Bronvermeldingen

**Thema:** Hoe documenten naar elkaar verwijzen (source tracking).

**Kernvragen:** Welke documenten fungeren als spin in het web (hebben de meeste bron-documenten of afgeleide versies via `document_bron_document`)?

Dit notebook verkent o.a. de silver-tabellen `document`, `document_bron_document`, `document_versie` en `document_publicatie_metadata`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

In [ ]:
query_df("DESCRIBE silver.document")

In [ ]:
query_df("DESCRIBE silver.document_bron_document")

In [ ]:
query_df("DESCRIBE silver.document_versie")

In [ ]:
query_df("DESCRIBE silver.document_publicatie_metadata")

## Onderwerp 1

### Welke documenten worden het vaakst als bron gebruikt door andere documenten?

In `document_bron_document` staat per document (`document_id`) naar welke bron-documenten (`bron_document_id`) verwezen wordt. Door te groeperen op `bron_document_id` en te tellen, zien we welke documenten het vaakst als bron dienen voor andere documenten. 

In [ ]:
meest_geciteerde_documenten = query_df("""
    SELECT
        dbd.bron_document_id,
        d.titel,
        d.soort,
        COUNT(*) AS aantal_verwijzingen
    FROM silver.document_bron_document AS dbd
    JOIN silver.document AS d
        ON dbd.bron_document_id = d.id
    GROUP BY dbd.bron_document_id, d.titel, d.soort
    ORDER BY aantal_verwijzingen DESC
    LIMIT 20
""")

meest_geciteerde_documenten

**Bevinding:** De meest gebruikte bron-documenten zijn vooral "Brief regering" en "Memorie van toelichting". Dit zijn dus type documenten die vaak als bron gebruikt worden voor andere stukken. Het document "Regels ter bescherming van de natuur (Wet...)" wordt het vaakst als bron gebruikt met 89 verwijzingen.

**Datakwaliteit:** Bij rij 5 staat een NaN als titel, terwijl het document wel 70 keer als bron wordt gebruikt. Dit betekent dat de titel ontbreekt in de brondata voor dit document. Het is een datakwaliteitsprobleem. 

In [ ]:
plt.figure(figsize=(10,10))
meest_geciteerde_documenten.plot.barh(x="titel", y="aantal_verwijzingen", title="Top 20 meest gebruikte bron-documenten")
plt.xlabel("Aantal verwijzingen")
plt.ylabel("")
plt.gca().invert_yaxis()
plt.show()

## Onderwerp 2

### Hoe vaak is er een bronvermelding?

Elke rij in `document_bron_document` is één bronvermelding (een document dat naar een ander document als bron verwijst). Door het aantal rijen in deze tabel te tellen, weten we hoe vaak er in totaal een bronvermelding voorkomt. Ook is het interessant om te weten bij hoeveel verschillende documenten er minstens één bronvermelding staat, ten opzichte van het totale aantal documenten. Dat geeft een beeld van hoe gebruikelijk bronvermelding is.

In [ ]:
aantal_bronvermeldingen = query_df("""
    SELECT
        COUNT(*) AS totaal_aantal_bronvermeldingen,
        COUNT(DISTINCT document_id) AS aantal_documenten_met_bronvermelding
    FROM silver.document_bron_document
""")

totaal_documenten = query_df("""
    SELECT COUNT(*) AS totaal_aantal_documenten
    FROM silver.document
""")

aantal_bronvermeldingen

In [ ]:
totaal_documenten

In [ ]:
documenten_met_meerdere_bronnen = query_df("""
    SELECT
        document_id,
        COUNT(*) AS aantal_bronnen
    FROM silver.document_bron_document
    GROUP BY document_id
    HAVING COUNT(*) > 1
""")

documenten_met_meerdere_bronnen

**Bevinding:** Van de 611.885 documenten hebben 123.887 (±20%) minstens één bronvermelding. Een controle (groeperen op `document_id` en filteren op `COUNT(*) > 1`) bevestigt dat geen enkel document meer dan één bronvermelding heeft; de relatie tussen een document en zijn bronvermelding is dus 1-op-1, niet 1-op-veel zoals je bij "bronvermeldingen" zou verwachten.

**Datakwaliteit:** Dit is opvallend: normaal gesproken verwijst een document vaak naar meerdere bronnen tegelijk (bijvoorbeeld een Kamerbrief die naar meerdere stukken verwijst). Dat dit hier niet voorkomt, kan wijzen op een beperking in hoe `document_bron_document` gevuld wordt tijdens het ETL-proces. Mogelijk wordt er per document maar één bronvermelding vastgelegd, ook als er meerdere bronnen zijn. 

In [ ]:
pd.Series({
    "Met bronvermelding": 123887,
    "Zonder bronvermelding": 611885 - 123887
}).plot.bar(title="Documenten met vs. zonder bronvermelding")
plt.ylabel("Aantal documenten")
plt.xticks(rotation=0)
plt.show()

## Onderwerp 3

### Welke soorten documenten worden het vaakst als bron gebruikt?

`document_bron_document` geeft aan welk document (`bron_document_id`) als bron dient. Om te zien wat voor soort document dit is, koppelen we aan `document` en groeperen we op de kolom `soort`. Dit laat zien welke soorten documenten (bijvoorbeeld Brief regering of Memorie van toelichting) het vaakst als bron worden gebruikt.

In [ ]:
soorten_bronnen = query_df("""
    SELECT
        d.soort,
        COUNT(*) AS aantal_keer_als_bron
    FROM silver.document_bron_document AS dbd
    JOIN silver.document AS d
        ON dbd.bron_document_id = d.id
    GROUP BY d.soort
    ORDER BY aantal_keer_als_bron DESC
    LIMIT 15
""")

soorten_bronnen

**Bevinding:** "Brief regering" is veruit het meest gebruikte soort document als bron (91.765 keer). Dit is ruim 10 keer zo vaak als nummer 2, "Antwoord schriftelijke vragen" (9.329 keer). Dit komt overeen met Onderwerp 1, waar "Brief regering" ook al vaak voorkwam als meest geciteerd document. Dit komt waarschijnlijk omdat kamerleden voortdurend reageren op brieven van de regering, dus het is te verwachten dat dit soort stukken vaak als bron worden gebruikt.

In [ ]:
plt.figure(figsize=(10,8))
soorten_bronnen.plot.barh(x="soort", y="aantal_keer_als_bron", title="Top 15 soorten documenten als bron")
plt.xlabel("Aantal keer als bron")
plt.ylabel("")
plt.gca().invert_yaxis()
plt.show()

## Onderwerp 4

### Hoeveel tijd zit er tussen de documentdatum en de daadwerkelijke publicatiedatum?

`document` bevat de `datum` van een document, en `document_publicatie_metadata` bevat de `publicatie_datum` waarop het daadwerkelijk gepubliceerd is (gekoppeld via `document_versie`). Door het verschil tussen deze twee datums te berekenen, zien we hoe snel documenten gepubliceerd worden nadat ze zijn opgesteld. Een negatief verschil (publicatie vóór de documentdatum) zou een datafout kunnen betekenen.

In [ ]:
publicatiegat = query_df("""
    SELECT
        d.id,
        d.titel,
        d.soort,
        d.datum,
        dpm.publicatie_datum,
        DATE_DIFF('day', d.datum, dpm.publicatie_datum) AS dagen_tot_publicatie
    FROM silver.document AS d
    JOIN silver.document_versie AS dv
        ON d.id = dv.document_id
    JOIN silver.document_publicatie_metadata AS dpm
        ON dv.id = dpm.document_versie_id
    WHERE d.datum IS NOT NULL
      AND dpm.publicatie_datum IS NOT NULL
""")

publicatiegat["dagen_tot_publicatie"].describe()

**Bevinding:** Gemiddeld zit er zo'n 1064 dagen (bijna 3 jaar) tussen de datum van het document en de datum waarop het echt gepubliceerd is. De mediaan ligt ook rond de 1060 dagen, dus het is niet zo dat een paar extreme gevallen het gemiddelde omhoog trekken, dit geldt gewoon voor de meeste documenten. Dat is best lang: je zou verwachten dat documenten vrij snel na het opstellen openbaar worden. Het verschil loopt uiteen van 0 tot wel 5345 dagen (bijna 15 jaar), dus het verschilt ook erg per document.

**Datakwaliteit:** een gemiddeld gat van bijna 3 jaar lijkt me niet kloppen voor een parlementair proces, waar je zou verwachten dat stukken redelijk snel openbaar worden. Dit zou kunnen betekenen dat `datum` of `publicatie_datum` niet goed wordt aangeleverd of verkeerd gekoppeld raakt tijdens het ETL-proces. Dit zou ik dus eerst verder willen uitzoeken voordat ik deze kolommen zou gebruiken voor echte analyses.

De grafiek laat een duidelijke piek zien vlak bij 0 dagen (ruim 70.000 documenten), gevolgd door een vrij vlakke verdeling tussen de 500 en 2500 dagen. Boven de 2500 dagen vallen bijna alle documenten weg, op een klein groepje na rond de 3000-4000 dagen. Dit suggereert dat er eigenlijk **twee verschillende groepen** documenten zijn: één die (bijna) direct gepubliceerd wordt, en een andere, grotere groep die er juist heel lang over doet. Dat versterkt het vermoeden dat hier een datakwaliteits- of ETL-probleem in het spel is, in plaats van één consistente publicatiewerkwijze.

In [ ]:
plt.figure(figsize=(10,6))
publicatiegat["dagen_tot_publicatie"].plot.hist(bins=30, title="Verdeling van het aantal dagen tot publicatie")
plt.xlabel("Aantal dagen tot publicatie")
plt.ylabel("Aantal documenten")
plt.show()